# Import the Dataframe

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import geopandas as gpd
import seaborn as sns

from matplotlib.colors import LogNorm
from scipy.stats import linregress, chi2_contingency, ttest_ind
from sklearn.metrics import confusion_matrix, accuracy_score

Read in the cleaned dataframe from the other notebook. 

**Reminder:** Our dataframe has 8352 data points and 32 variables per data point.

In [ ]:
df = pd.read_excel(r"C:\Users\n_jac\RadStar\cleaned_data.xlsx")
df

Create some subsets of the data based on the 5 major latitude zones so that we can observe trends between the different zones later.

In [ ]:
df_arctic = df[df["lat"] > 66.5].copy()
df_antarctic = df[df["lat"] < -66.5].copy()
df_tropical = df[df["lat"].between(-23.5, 23.5)].copy()
df_stz = df[df["lat"].between(-66.5, -23.5, inclusive="neither")].copy()
df_ntz = df[df["lat"].between(23.5, 66.5, inclusive="neither")].copy()

In [ ]:
url = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
world = gpd.read_file(url)

# Exploration of the X-Ray Radiation Data

In our meeting on June 18th with Dr. Voss and NearSpace Launch, they directed our focus onto going deeper into the x-ray radiation data. This was the first time that a NearSpace Launch satellite collected x-ray radiation data, so they were unsure how well the collection software was working and general trends among the data.

## Similar Trends Across All 4 X-Ray Detectors

The first step in checking the validity of the x-ray data columns was checking to see if we see similar trends between all 4 of the sensors. These sensors have different thresholds, so if the different different columns show hotspots in different areas than their is likely an issue in the data collection process. 

In [ ]:
plt.figure(figsize=(15, 6))

plt.boxplot([
    df['xray0_ps'],
    df['xray1_ps'],
    df['xray2_ps'],
    df['xray3_ps']
])

plt.xticks(
    [1, 2, 3, 4],
    ['xray0','xray1','xray2','xray3']
)

plt.yscale('log')
plt.ylabel('Count per Second (Log Scale)')
plt.title('X-Ray Radiation Distributions (per second) Across Different Thresholds')
plt.show()

In [ ]:
df[['xray0_ps', 'xray1_ps', 'xray2_ps', 'xray3_ps']].describe()

This shows us that going forward the higher threshold x-ray radiation data columns are not very beneficial in data analysis. Since the over 75% of all three of the higher threshold columns collected 0 xray radiation waves. This fact may also be hinting at the thresholds being set too high, creating a sensor that can not collect sufficient data since xray radiation waves very rarely reach the required threshold. 

In [ ]:
df_x0 = df[df["xray0_ps"] != 0].copy()
df_x1 = df[df["xray1_ps"] != 0].copy()
df_x2 = df[df["xray2_ps"] != 0].copy()
df_x3 = df[df["xray3_ps"] != 0].copy()

When we take out the rows where the sensor registered 0 x-ray radiation in that location, we are left with 
- 7925 data points in 'xray0' (removed 427 data points)
- 1567 data points in 'xray1' (removed 6785 data points)
- 1019 data points in 'xray2' (removed 7333 data points)
- 307 data points in 'xray3' (removed 8045 data points)

In [ ]:
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(24, 12))

world.plot(ax=ax1, color="lightgray")
sc1 = ax1.scatter(
    df_x0["lon"],
    df_x0["lat"],
    c=df_x0["xray0_ps"],
    s=5,
    norm=LogNorm()
)
ax1.set_title("X-Ray0 per Second (non-zero values)",fontsize="xx-large")
ax1.set_xlabel("Longitude")
ax1.set_ylabel("Latitude")
plt.colorbar(sc1, ax=ax1)

world.plot(ax=ax2, color="lightgray")
sc2 = ax2.scatter(
    df_x1["lon"],
    df_x1["lat"],
    c=df_x1["xray1_ps"],
    s=5,
    norm=LogNorm()
)
ax2.set_title("X-Ray1 per Second (non-zero values)",fontsize="xx-large")
ax2.set_xlabel("Longitude")
ax2.set_ylabel("Latitude")
plt.colorbar(sc2, ax=ax2)

world.plot(ax=ax3, color="lightgray")
sc3 = ax3.scatter(
    df_x2["lon"],
    df_x2["lat"],
    c=df_x2["xray2_ps"],
    s=5,
    norm=LogNorm()
)
ax3.set_title("X-Ray2 per Second (non-zero values)",fontsize="xx-large")
ax3.set_xlabel("Longitude")
ax3.set_ylabel("Latitude")
plt.colorbar(sc3, ax=ax3)

world.plot(ax=ax4, color="lightgray")
sc4 = ax4.scatter(
    df_x3["lon"],
    df_x3["lat"],
    c=df_x3["xray3_ps"],
    s=5,
    norm=LogNorm()
)
ax4.set_title("X-Ray3 per Second (non-zero values)",fontsize="xx-large")
ax4.set_xlabel("Longitude")
ax4.set_ylabel("Latitude")
plt.colorbar(sc4, ax=ax4)

plt.tight_layout()
plt.show()

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(30, 6))

sc1 = ax1.scatter(
    df["lon"],
    df["lat"],
    c=df["xray0_ps"],
    s=5
)
ax1.set_title("X-Ray0 per Second")
ax1.set_xlabel("Longitude")
ax1.set_ylabel("Latitude")
plt.colorbar(sc1, ax=ax1)

sc2 = ax2.scatter(
    df_x0["lon"],
    df_x0["lat"],
    c=df_x0["xray0_ps"],
    s=5
)
ax2.set_title("X-Ray0 per Second (non-zero values)")
ax2.set_xlabel("Longitude")
ax2.set_ylabel("Latitude")
plt.colorbar(sc2, ax=ax2)

sc3 = ax3.scatter(
    df_x0["lon"],
    df_x0["lat"],
    c=df_x0["xray0_ps"],
    s=5,
    norm=LogNorm()
)
ax3.set_title("X-Ray0 per Second (non-zero values) ")
ax3.set_xlabel("Longitude")
ax3.set_ylabel("Latitude")
plt.colorbar(sc3, ax=ax3)

plt.tight_layout()
plt.show()

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(30, 6))

sc1 = ax1.scatter(
    df["lon"],
    df["lat"],
    c=df["xray1_ps"],
    s=5
)
ax1.set_title("X-Ray1 per Second")
ax1.set_xlabel("Longitude")
ax1.set_ylabel("Latitude")
plt.colorbar(sc1, ax=ax1)

sc2 = ax2.scatter(
    df_x1["lon"],
    df_x1["lat"],
    c=df_x1["xray1_ps"],
    s=5
)
ax2.set_title("X-Ray1 per Second")
ax2.set_xlabel("Longitude")
ax2.set_ylabel("Latitude")
plt.colorbar(sc2, ax=ax2)

sc3 = ax3.scatter(
    df_x1["lon"],
    df_x1["lat"],
    c=df_x1["xray1_ps"],
    s=5,
    norm=LogNorm()
)
ax3.set_title("X-Ray1 per Second")
ax3.set_xlabel("Longitude")
ax3.set_ylabel("Latitude")
plt.colorbar(sc3, ax=ax3)

plt.tight_layout()
plt.show()

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(30, 6))

sc1 = ax1.scatter(
    df["lon"],
    df["lat"],
    c=df["xray2_ps"],
    s=5
)
ax1.set_title("X-Ray2 per Second")
ax1.set_xlabel("Longitude")
ax1.set_ylabel("Latitude")
plt.colorbar(sc1, ax=ax1)

sc2 = ax2.scatter(
    df_x2["lon"],
    df_x2["lat"],
    c=df_x2["xray2_ps"],
    s=5
)
ax2.set_title("X-Ray2 per Second")
ax2.set_xlabel("Longitude")
ax2.set_ylabel("Latitude")
plt.colorbar(sc2, ax=ax2)

sc3 = ax3.scatter(
    df_x2["lon"],
    df_x2["lat"],
    c=df_x2["xray2_ps"],
    s=5,
    norm=LogNorm()
)
ax3.set_title("X-Ray2 per Second")
ax3.set_xlabel("Longitude")
ax3.set_ylabel("Latitude")
plt.colorbar(sc3, ax=ax3)

plt.tight_layout()
plt.show()

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(30, 6))

sc1 = ax1.scatter(
    df["lon"],
    df["lat"],
    c=df["xray3_ps"],
    s=5
)
ax1.set_title("X-Ray3 per Second")
ax1.set_xlabel("Longitude")
ax1.set_ylabel("Latitude")
plt.colorbar(sc1, ax=ax1)

sc2 = ax2.scatter(
    df_x3["lon"],
    df_x3["lat"],
    c=df_x3["xray3_ps"],
    s=5
)
ax2.set_title("X-Ray3 per Second")
ax2.set_xlabel("Longitude")
ax2.set_ylabel("Latitude")
plt.colorbar(sc2, ax=ax2)

sc3 = ax3.scatter(
    df_x3["lon"],
    df_x3["lat"],
    c=df_x3["xray3_ps"],
    s=5,
    norm=LogNorm()
)
ax3.set_title("X-Ray3 per Second")
ax3.set_xlabel("Longitude")
ax3.set_ylabel("Latitude")
plt.colorbar(sc3, ax=ax3)

plt.tight_layout()
plt.show()

We can see in the graphs in the first figure that the two major hotspots of the the lowest threshold sensor was the Arctic Circle and the South Atlantic Anomaly. Then when we look at the higher threshold sensors, we see that while most of the data points go away since they recorded 0 radiation. However, most of the locations that the higher threshold sensors did pick up radiation occured in these two areas. Thus, we found that the 4 sensors did have similiar trends and thus did not flag an issue with the sensors functionality. 

## X-Ray and Electron Radiation

One potential issue that Dr. Voss and NearSpace Launch raised on the functionality of their x-ray radiation sensors was the possibility that electron radiation was leaking into the x-ray sensor and being counted as x-ray radiation. Thus looking into that possiblity through relationship between the x-ray radiation and electron radiation columns.

### Linear Regression t-Tests

In a linear regression t-test, our hypotheses are...

- $H_0: \beta = 0$
- $H_a: \beta \neq 0$

then we will reject the null hypothesis if our p-value is less than 0.05

In [ ]:
def lin_reg_ttest(x,y):
    for i in x:
        for j in y:
            result = linregress(df[i],df[j])
            slope = result.slope
            intercept = result.intercept
            r_squared = result.rvalue**2
            p_value = result.pvalue
            t_stat = result.slope / result.stderr
            
            y_fit = slope * df[i] + intercept
            
            plt.figure(figsize=(8, 6))
            plt.scatter(df[i], df[j], s=5, alpha=0.5)
            plt.plot(df[i], y_fit, color="red")
            
            plt.xlabel(i)
            plt.ylabel(j)
            plt.title(i + ' vs ' + j)
            
            plt.show()
            
            print(f"Slope: {slope:.6f}")
            print(f"Intercept: {intercept:.6f}")
            print(f"R²: {r_squared:.4f}")
            print(f"t-statistic: {t_stat:.4f}")
            print(f"p-value: {p_value:.6g}")

In [ ]:
x = df["xray0"]
y = df["electron0"]

result = linregress(x, y)

slope = result.slope
intercept = result.intercept
r_squared = result.rvalue**2
p_value = result.pvalue
t_stat = result.slope / result.stderr

y_fit = slope * x + intercept

plt.figure(figsize=(8, 6))
plt.scatter(x, y, s=5, alpha=0.5)
plt.plot(x, y_fit, color="red")

plt.xlabel("xray0")
plt.ylabel("electron0")
plt.title("xray0 vs electron0")

plt.show()

print(f"Slope: {slope:.6f}")
print(f"Intercept: {intercept:.6f}")
print(f"R²: {r_squared:.4f}")
print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.6g}")

**Linear Regression t-Test between 'xray0' and 'electron0'**

We are running a linear regression t-test to see if there is a relationship between the 'xray0' and 'electron0' columns. Since the test yields a p-value that is extremely small, we will reject the null hypothesis and conclude that the slope is statistically significant. However, the statistically signifant slope is very gradual and inverted (-0.03). This means that for every additional x-ray wave picked up by our sensor, our electron radiation sensor should pick up 0.03 less electrons. Another thing to note is that our $R^2$ value is also extremely small (0.0099), which means that the linear regression line only accounts for 0.99% of the variation in the electron radiation. 

Some things to note from this test are...
1. We expected to see a positive relationship not an inverse one, which makes the result very interesting
2. While statistically significant the slope of the regression line accounts for a very small percentage of the variation in electron radiation

In [ ]:
xlab = ['xray0','xray1','xray2','xray3']
ylab = ['electron0','electron1']

lin_reg_ttest(xlab, ylab)

As we moved past the lowest threshold x-ray sensor, we started to see the positive correlation between the two sensors that Dr. Voss and NearSpace Launch was worried about. Also, all 8 combinations of the linear regression t-tests yielded a low p-value and thus a statistically significant slope. However, the greatest $R^2$ value was just over 0.03, and thus none of the linear regression lines accounted for a large portion of the variation in the electron sensors. Therefore, we can't draw any strong conclusions from these tests as to if the electron radiation was leaking into the x-ray sensors and skewing the data. If this is something that NearSpace Launch finds concerning, then they should try to run some additional tests and see if the electron radiation leakage is really occuring in their x-ray sensors.

### Mix of X-Ray and Electron Radiation in X-Ray Sensors

In [ ]:
df["electron0 + xray1"] = df["electron0"] + df["xray1"]
df["electron0 + xray2"] = df["electron0"] + df["xray2"]
df["electron0 + xray3"] = df["electron0"] + df["xray3"]
df["electron1 + xray1"] = df["electron1"] + df["xray1"]
df["electron1 + xray2"] = df["electron1"] + df["xray2"]
df["electron1 + xray3"] = df["electron1"] + df["xray3"]

In [ ]:
xlabs = ['xray0','xray1','xray2','xray3']
ylabs = ['electron0 + xray1','electron0 + xray2','electron0 + xray3','electron1 + xray1','electron1 + xray2','electron1 + xray3']

lin_reg_ttest(xlabs,ylabs)

All of the linear regression t-tests between the x-ray radiation columns and the new columns the was the sum of higher threshold x-ray and/or electron radiation columns gave similar results as the simple x-ray vs electron radiation columns. That is that the slope is statistically significance due to low p-values, but also does not account for much of the variation in the y-axis sums due to the low $R^2$ values. 

## X-Ray During Day vs Night 

### Check Validity of Column Values

NearSpace Launch informed us that were uncertain as to the validity of the "in_shadow" column of their data set. I was able to find a library that allowed us to make a function that uses the timestamp, latitude, longitude, and altitude data of the satellite to check if the Earth was in between the sun and our satellite.

In [ ]:
from astropy.time import Time
from astropy.coordinates import EarthLocation, get_sun
import astropy.units as u

EARTH_RADIUS = 6371.0  # km

def is_in_earth_shadow(lat, lon, alt_km, timestamp):
    # Convert to astropy EarthLocation
    location = EarthLocation(lat=lat*u.deg, lon=lon*u.deg, height=alt_km*u.km)
    
    # Get Sun position in ITRS (ECEF) coordinates
    t = Time(timestamp)
    sun_itrs = get_sun(t).transform_to('itrs')
    sun_vector = np.array([sun_itrs.x.to(u.km).value,
                           sun_itrs.y.to(u.km).value,
                           sun_itrs.z.to(u.km).value])
    
    # Satellite position in ECEF
    sat_vector = np.array([location.x.to(u.km).value,
                           location.y.to(u.km).value,
                           location.z.to(u.km).value])
    
    # Vector from satellite to Sun
    sat_to_sun = sun_vector - sat_vector
    
    # Project satellite position onto Sun vector
    t_proj = -np.dot(sat_vector, sat_to_sun) / np.dot(sat_to_sun, sat_to_sun)
    
    if t_proj < 0:
        return False  # Sun is "behind" satellite
    
    # Closest approach distance from Earth's center to Sun-satellite line
    closest_dist = np.linalg.norm(sat_vector + t_proj * sat_to_sun)
    
    return closest_dist < EARTH_RADIUS

In [ ]:
df["shadow_check"] = df.apply(
    lambda row: is_in_earth_shadow(
        row["lat"],
        row["lon"],
        row["alt"],
        row["timestamp"]
    ),
    axis=1
)

In [ ]:
df["shadow_check"] = df["shadow_check"].astype(int)
df = df.dropna(subset=["in_shadow"])

In [ ]:
cm = confusion_matrix(df["shadow_check"],df["in_shadow"],labels=[0, 1])
print(cm)

accuracy = accuracy_score(df["shadow_check"],df["in_shadow"])
print(f"Accuracy: {accuracy:.4f}")

When we run our function on our data and then compare the function's in shaodw value with the original in shadow values, we can get that 99.96% of the values are the same. In fact, there were only 3 values that differed between the function and the original, and all 3 of these values were deemed not in shadow in the original column but and deemed in shadow using the function. We decided to just use the original in shadow values going forward since the similarity was so similar between the two columns. 

### Exploratory

In [ ]:
day = df[df['in_shadow']==0].copy()
night = df[df['in_shadow']==1].copy()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))

world.plot(ax=ax1, color="lightgray")
ax1.scatter(
    day["lon"],
    day["lat"],
    c='black',
    s=5
)
ax1.set_title("Daytime Locations")

world.plot(ax=ax2, color="lightgray")
ax2.scatter(
    night["lon"],
    night["lat"],
    c='black',
    s=5
)
ax2.set_title("Nighttime Locations")

plt.tight_layout()
plt.show()

These two graphs show us the locations in which data was registered when the satellite was both in the Earth's shadow and when it was not. One concerning thing that we noticed from these graphs is the lack of in shadow data from both the Arctic and Antarctic Circles. 

In [ ]:
plt.figure(figsize=(8, 6))

plt.boxplot([
    day['xray0_ps'],
    night['xray0_ps']
])

plt.xticks(
    [1, 2],
    ['In Sun','In Shadow']
)

plt.yscale('log')
plt.ylabel('Count per Second (Log Scale)')
plt.title('X-Ray Radiation Distributions (per second)')
plt.show()

In [ ]:
print('IN SUN')
print(day['xray0_ps'].describe())

print()
print('IN SHADOW')
print(night['xray0_ps'].describe())

In [ ]:
plt.figure(figsize=(15, 6))

plt.boxplot([
    day['xray0_ps'],
    night['xray0_ps'],
    day['electron0_ps'],
    night['electron0_ps'],
    day['proton0_ps'],
    night['proton0_ps']
])

plt.xticks(
    [1, 2, 3, 4, 5, 6],
    ['In Sun X-Ray','In Shadow X-Ray','In Sun Electron','In Shadow Electron','In Sun Proton','In Shadow Proton']
)

plt.yscale('log')
plt.ylabel('Count per Second (Log Scale)')
plt.title('Electron and Proton Radiation Distributions (per second)')
plt.show()

In [ ]:
print('ELECTRON IN SUN')
print(day['electron0_ps'].describe())

print()
print('ELECTRON IN SHADOW')
print(night['electron0_ps'].describe())

In [ ]:
print('PROTON IN SUN')
print(day['proton0_ps'].describe())

print()
print('PROTON IN SHADOW')
print(night['proton0_ps'].describe())

This shows us that there is a large difference in the all three types of radiation when our satellite can see the sun versus when the satellite is in the Earth's shadow, with all three types of radiation being higher when the satellite is in the sun. However, we will need to do more data analysis on this in order to prove that this difference is not due to some confounding variables. For example, in the first visual in this secton we saw the locations of the in sun versus in shadow data points, and in that visual we noted that all of the in shadow data points fell in the middle more tropical regions of the earth with very few in shadow data points in the Arctic and Antarctic Circles. 

#### Welsh's Two-Sample t-Tests

In Welch's two-sample t-test, our hypotheses are...

- $H_0: \bar{x}_{day} = \bar{x}_{night}$
- $H_a: \bar{x}_{day} \neq \bar{x}_{night}$

then we will reject the null hypothesis if our p-value is less than 0.05.

In [ ]:
t_stat, p_value = ttest_ind(day['xray0_ps'], night['xray0_ps'], equal_var=False)

print("t-statistic:", t_stat)
print("p-value:", p_value)

First we take the full dataset and look at the x-ray radiation distributions between when the satellite is in direct sunlight and when it is in the Earth's shadow. The Welsh's Two-Sample t_Test on the resulting distribution yields a p-value that is virtually 0, thus we reject the null hypothesis and conclude that the mean amount of x-ray radiation in the sun is different then the mean amount of radiation in the Earth's shadow. 

In [ ]:
t_stat, p_value = ttest_ind(day['electron0_ps'], night['electron0_ps'], equal_var=False)

print("t-statistic:", t_stat)
print("p-value:", p_value)

In [ ]:
t_stat, p_value = ttest_ind(day['proton0_ps'], night['proton0_ps'], equal_var=False)

print("t-statistic:", t_stat)
print("p-value:", p_value)

In [ ]:
df["region"] = np.select(
    [
        df["lat"] > 66.5,
        df["lat"] < -66.5
    ],
    [
        "Arctic",
        "Antarctic"
    ],
    default="Tropical"
)

In [ ]:
table = pd.crosstab(
    df["region"],
    df["in_shadow"]
)

print(table)

In [ ]:
day_tropical = day[day["lat"].between(-66.5, 66.5)].copy()
night_tropical = night[night["lat"].between(-66.5, 66.5)].copy()
day_arctic = day[day["lat"] > 66.5].copy()
night_arctic = night[night["lat"] > 66.5].copy()
day_antarctic = day[day["lat"] < -66.5].copy()
night_antarctic = night[night["lat"] < -66.5].copy()

In [ ]:
t_stat, p_value = ttest_ind(day_tropical['xray0_ps'], night_tropical['xray0_ps'], equal_var=False)

print("t-statistic:", t_stat)
print("p-value:", p_value)

I believe that this central region of the globe will give us the most accurate test as there is a substantial amount of data for both the day and night distributions. In this Welsh Two-Sample t-Test, we also get a p-value that is virtually 0 and thus will again reject the null hypothesis in favor of there being differing means between when the satellite is in the sun versus in the Earth's shadow. 

In [ ]:
t_stat, p_value = ttest_ind(day_arctic['xray0_ps'], night_arctic['xray0_ps'], equal_var=False)

print("t-statistic:", t_stat)
print("p-value:", p_value)

In [ ]:
t_stat, p_value = ttest_ind(day_antarctic['xray0_ps'], night_antarctic['xray0_ps'], equal_var=False)

print("t-statistic:", t_stat)
print("p-value:", p_value)

While the tests might be slightly inaccurate in the Arctic and Antarctic Circles due to the lack of in shadow data points, I figured that I would run the tests anyways. Once again for both the Arctic and Antarctic Circles, we get p-values that is virtually 0 and thus will again reject the null hypothesis in favor of there being differing means between when the satellite is in the sun versus in the Earth's shadow. 

In [ ]:
x = day["lat"]
y = day["xray0_ps"]

result = linregress(x, y)

slope = result.slope
intercept = result.intercept
r_squared = result.rvalue**2

y_fit = slope * x + intercept

plt.figure(figsize=(12, 6))
plt.scatter(x, y, s=5, alpha=0.5)
plt.plot(x, y_fit)

plt.xlabel("Latitude")
plt.ylabel("X-Ray 0")
plt.title("X-Ray vs Latitude (Daytime)")

plt.show()

print(f"Slope: {slope}")
print(f"Intercept: {intercept}")
print(f"R²: {r_squared:.4f}")

In [ ]:
x = night["lat"]
y = night["xray0_ps"]

result = linregress(x, y)

slope = result.slope
intercept = result.intercept
r_squared = result.rvalue**2

y_fit = slope * x + intercept

plt.figure(figsize=(12, 6))
plt.scatter(x, y, s=5, alpha=0.5)
plt.plot(x, y_fit)

plt.xlabel("Latitude")
plt.ylabel("X-Ray 0")
plt.title("X-Ray vs Latitude (Nighttime)")

plt.show()

print(f"Slope: {slope}")
print(f"Intercept: {intercept}")
print(f"R²: {r_squared:.4f}")

In [ ]:
x = day["lat"]
y = day["xray0_ps"]

# Fit quadratic: y = ax² + bx + c
a, b, c = np.polyfit(x, y, 2)

# Create smooth curve for plotting
x_fit = np.linspace(x.min(), x.max(), 1000)
y_fit = a * x_fit**2 + b * x_fit + c

# Calculate R²
y_pred = a * x**2 + b * x + c

ss_res = np.sum((y - y_pred)**2)
ss_tot = np.sum((y - np.mean(y))**2)
r_squared = 1 - ss_res / ss_tot

# Plot
plt.figure(figsize=(12, 6))
plt.scatter(x, y, s=5, alpha=0.5)
plt.plot(x_fit, y_fit, linewidth=2)

plt.xlabel("Latitude")
plt.ylabel("X-Ray 0")
plt.title("X-Ray vs Latitude (Nighttime)")

plt.show()

print(f"y = {a:.6e}x² + {b:.6e}x + {c:.6e}")
print(f"R² = {r_squared:.4f}")